In [1]:
import pandas as pd

# Uploading dataset

In [2]:
train = pd.read_parquet("/content/train-00000-of-00001.parquet")
test = pd.read_parquet("/content/test-00000-of-00001.parquet")
validation = pd.read_parquet("/content/validation-00000-of-00001.parquet")

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67349 entries, 0 to 67348
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   idx       67349 non-null  int32 
 1   sentence  67349 non-null  object
 2   label     67349 non-null  int64 
dtypes: int32(1), int64(1), object(1)
memory usage: 1.3+ MB


In [6]:
train.shape

(67349, 3)

In [7]:
validation.shape

(872, 3)

In [8]:
print("Information about train dataset")
print(train.info())

Information about train dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67349 entries, 0 to 67348
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   idx       67349 non-null  int32 
 1   sentence  67349 non-null  object
 2   label     67349 non-null  int64 
dtypes: int32(1), int64(1), object(1)
memory usage: 1.3+ MB
None


In [9]:
print("Information about validation dataset")
print(validation.info())

Information about validation dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 872 entries, 0 to 871
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   idx       872 non-null    int32 
 1   sentence  872 non-null    object
 2   label     872 non-null    int64 
dtypes: int32(1), int64(1), object(1)
memory usage: 17.2+ KB
None


In [12]:
train.head(5)

,idx,sentence,label
0,0,hide new secretions from the parental units,0
1,1,"contains no wit , only labored gags",0
2,2,that loves its characters and communicates som...,1
3,3,remains utterly satisfied to remain the same t...,0
4,4,on the worst revenge-of-the-nerds clichés the ...,0


In [13]:
train_text = train["sentence"]
train_label = train["label"]

val_text = validation["sentence"]
val_label = validation["label"]

In [14]:
from numpy import vectorize
from sklearn.feature_extraction.text import TfidfVectorizer

vectorize = TfidfVectorizer(stop_words="english", max_df=0.7)

train_vectors = vectorize.fit_transform(train_text)
val_vectors = vectorize.transform(val_text)

In [15]:
import numpy as np

positive_vector = train_vectors[np.array(train_label) == 1].mean(axis=0)
negative_vector = train_vectors[np.array(train_label) == 0].mean(axis=0)

In [16]:
positive_vector = np.asarray(positive_vector)
negative_vector = np.asarray(negative_vector)

In [17]:
print(train_vectors.shape)
print(val_vectors.shape)
print(positive_vector.shape)
print(negative_vector.shape)

(67349, 13491)
(872, 13491)
(1, 13491)
(1, 13491)


# TF-IDF + Cosine Similarity

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

positive_similarity = cosine_similarity(val_vectors, positive_vector)
negative_similarity = cosine_similarity(val_vectors, negative_vector)

In [19]:
predictions_cos = (positive_similarity > negative_similarity).astype(int)

In [20]:
from sklearn.metrics import accuracy_score, f1_score

f1_cos = f1_score(val_label, predictions_cos)

accuracy_cos = accuracy_score(val_label, predictions_cos)

print("TF-IDF + Cosine Similarity Results")
print("Accuracy:", accuracy_cos)
print("F1:", f1_cos)

TF-IDF + Cosine Similarity Results
Accuracy: 0.7477064220183486
F1: 0.7544642857142857


In [21]:
from sklearn.metrics import classification_report
print("TF-IDF + Cosine Similarity Results")
print(classification_report(val_label, predictions_cos, target_names=["Negative", "Positive"]))

TF-IDF + Cosine Similarity Results
              precision    recall  f1-score   support

    Negative       0.75      0.73      0.74       428
    Positive       0.75      0.76      0.75       444

    accuracy                           0.75       872
   macro avg       0.75      0.75      0.75       872
weighted avg       0.75      0.75      0.75       872



# TF-IDF + Logistic Regression

In [22]:
from sklearn.linear_model import LogisticRegression

logReg = LogisticRegression()
logReg.fit(train_vectors, train_label)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [23]:
predictions_logReg = logReg.predict(val_vectors)

In [24]:
f1_logReg = f1_score(val_label, predictions_logReg)

accuracy_logReg = accuracy_score(val_label, predictions_logReg)

print("TF-IDF + Logistic Regression Results")
print("Accuracy:", accuracy_logReg)
print("F1:", f1_logReg)

TF-IDF + Logistic Regression Results
Accuracy: 0.8038990825688074
F1: 0.8209424083769633


In [25]:
print("TF-IDF + Logistic Regression Results")
print(classification_report(val_label, predictions_logReg, target_names=["Negative", "Positive"]))

TF-IDF + Logistic Regression Results
              precision    recall  f1-score   support

    Negative       0.86      0.72      0.78       428
    Positive       0.77      0.88      0.82       444

    accuracy                           0.80       872
   macro avg       0.81      0.80      0.80       872
weighted avg       0.81      0.80      0.80       872



# TF-IDF + Support Vector Machine

In [27]:
from sklearn.svm import SVC

In [ ]:
svm = SVC()
svm.fit(train_vectors, train_label)

In [ ]:
predictions_svm = svm.predict(val_vectors)

In [ ]:
f1_svm = f1_score(val_label, predictions_svm)

accuracy_svm = accuracy_score(val_label, predictions_svm)
print("TF-IDF + Support Vector Machine Results")
print("Accuracy:", accuracy_svm)
print("F1:", f1_svm)

TF-IDF + Support Vector Machine Results
Accuracy: 0.7993119266055045
F1: 0.8175182481751825


In [ ]:
print("TF-IDF + Support Vector Machine Results")
print(classification_report(val_label, predictions_svm, target_names=["Negative", "Positive"]))

# BERT

In [4]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(validation)

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [6]:
def tokenize_function(dataset):
    return tokenizer(
        dataset["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128)

In [7]:
train_tokenized = train_dataset.map(tokenize_function)
val_tokenized = val_dataset.map(tokenize_function)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [8]:
print(train_tokenized.column_names)

['idx', 'sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [10]:
train_tokenized.data

InMemoryTable
idx: int32
sentence: string
label: int64
input_ids: list<item: int32>
  child 0, item: int32
token_type_ids: list<item: int8>
  child 0, item: int8
attention_mask: list<item: int8>
  child 0, item: int8
----
idx: [[0,1,2,3,4,...,995,996,997,998,999],[1000,1001,1002,1003,1004,...,1995,1996,1997,1998,1999],...,[66000,66001,66002,66003,66004,...,66995,66996,66997,66998,66999],[67000,67001,67002,67003,67004,...,67344,67345,67346,67347,67348]]
sentence: [["hide new secretions from the parental units ","contains no wit , only labored gags ","that loves its characters and communicates something rather beautiful about human nature ","remains utterly satisfied to remain the same throughout ","on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ",...,"you wish you were at home watching that movie instead of in the theater watching this one ","'s no point in extracting the bare bones of byatt 's plot for purposes of bland hollywood romance ","underdeveloped ","t

In [11]:
train_tokenized = train_tokenized.remove_columns(["idx", "sentence"])
val_tokenized = val_tokenized.remove_columns(["idx", "sentence"])

In [12]:
print(train_tokenized.column_names)

['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [13]:
from transformers import AutoModelForSequenceClassification
bert = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
train_dataset

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 67349
})

In [16]:
val_dataset

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 872
})

In [17]:
train_tokenized

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 67349
})

In [21]:
val_tokenized

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 872
})

In [22]:
bert

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [23]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(eval_pred):
    prediction_bert, labels = eval_pred
    prediction_bert = np.argmax(prediction_bert, axis=1)
    return {
        "accuracy": accuracy_score(labels, prediction_bert),
        "f1": f1_score(labels, prediction_bert)
    }

In [24]:
from transformers import TrainingArguments
training_argument = TrainingArguments(
    output_dir = "./bert_sst2",
    num_train_epochs = 2,
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    eval_strategy = "epoch",
    save_strategy = "no",
    report_to = "none")

In [25]:
from transformers import Trainer
trainer = Trainer(
    model = bert,
    args = training_argument,
    train_dataset = train_tokenized,
    eval_dataset = val_tokenized,
    compute_metrics = compute_metrics)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
